In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.exchange_rates")
 
# Display the data
display(df_customers)

In [0]:
from pyspark.sql.functions import to_date, date_format, col

# Read exchange_rates table
df_exchange_rates = spark.table("bronze_catalog.bronze_stage_schema.exchange_rates")

# Convert date string to proper date format, then format to yyyy-MM-dd
df_exchange_rates_formatted = df_exchange_rates.withColumn(
    "date",
    date_format(to_date(col("date"), "yyyy/M/d"), "yyyy-MM-dd")
)

# Display the formatted data
print("Exchange rates with formatted date:")
display(df_exchange_rates_formatted)

print(f"\nTotal rows: {df_exchange_rates_formatted.count()}")

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.exchange_rates where date = '2024-01-14';

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")
 
# Display the data
display(df_customers)

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.invoice_line_items where product_id = 'P059';

In [0]:
from pyspark.sql.functions import col, count, when

# Read invoice_line_items table
df_line_items = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")

# Count total rows and NULL discount rows
total_rows = df_line_items.count()
null_discount_count = df_line_items.filter(col("discount").isNull()).count()
non_null_discount_count = df_line_items.filter(col("discount").isNotNull()).count()

print("=" * 60)
print("NULL Value Analysis for 'discount' column")
print("=" * 60)
print(f"Total rows in invoice_line_items: {total_rows}")
print(f"Rows with NULL discount: {null_discount_count}")
print(f"Rows with non-NULL discount: {non_null_discount_count}")
print(f"NULL percentage: {(null_discount_count/total_rows)*100:.2f}%")
print("=" * 60)

# Show sample records with NULL discount
print("\nSample records with NULL discount values:")
df_line_items.filter(col("discount").isNull()).show(20, truncate=False)

# Show summary statistics
print("\nDiscount column summary:")
df_line_items.select(
    count("*").alias("total_records"),
    count("discount").alias("non_null_discounts"),
    count(when(col("discount").isNull(), 1)).alias("null_discounts")
).show()

In [0]:
# Sample data from user's dataset
print("=" * 70)
print("Calculating NULL discount value from sample data (INV000001)")
print("=" * 70)

# Line item L0000002 (has discount)
quantity_L2 = 8
unit_price_L2 = 337
discount_L2 = 0.05
line_total_L2 = quantity_L2 * unit_price_L2 * (1 - discount_L2)
print(f"\nL0000002: {quantity_L2} × {unit_price_L2} × (1 - {discount_L2}) = {line_total_L2:.2f}")

# Line item L0000001 (NULL discount - to be calculated)
quantity_L1 = 6
unit_price_L1 = 375
print(f"L0000001: {quantity_L1} × {unit_price_L1} × (1 - discount_L1) = ?")

# Payment information
payment_amount = 3579
print(f"\nTotal payment amount: {payment_amount}")

# Calculate the missing discount
# payment_amount = line_total_L1 + line_total_L2
# payment_amount = (quantity_L1 * unit_price_L1 * (1 - discount_L1)) + line_total_L2
# Solving for discount_L1:

line_total_L1_needed = payment_amount - line_total_L2
print(f"\nL0000001 line total needed: {payment_amount} - {line_total_L2:.2f} = {line_total_L1_needed:.2f}")

line_total_before_discount_L1 = quantity_L1 * unit_price_L1
print(f"L0000001 before discount: {quantity_L1} × {unit_price_L1} = {line_total_before_discount_L1}")

discount_multiplier = line_total_L1_needed / line_total_before_discount_L1
discount_L1 = 1 - discount_multiplier

print("\n" + "=" * 70)
print(f"CALCULATED DISCOUNT for L0000001: {discount_L1:.4f} ({discount_L1*100:.2f}%)")
print("=" * 70)

# Verification
verify_L1 = quantity_L1 * unit_price_L1 * (1 - discount_L1)
verify_total = verify_L1 + line_total_L2
print(f"\nVerification:")
print(f"L0000001: {quantity_L1} × {unit_price_L1} × (1 - {discount_L1:.4f}) = {verify_L1:.2f}")
print(f"L0000002: {line_total_L2:.2f}")
print(f"Total: {verify_L1:.2f} + {line_total_L2:.2f} = {verify_total:.2f}")
print(f"Expected payment: {payment_amount}")
print(f"Difference: {abs(verify_total - payment_amount):.2f}")

if abs(verify_total - payment_amount) < 0.01:
    print("\n✓ Calculation matches the payment amount!")
else:
    print("\n⚠ Warning: Calculation does not match payment amount.")
    print("   This could mean:")
    print("   - There are additional line items not shown in the sample")
    print("   - There are other charges/fees not accounted for")
    print("   - The discount NULL should be replaced with 0 (no discount)")

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Get all line items for invoice INV000001
df_invoice_lines = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items") \
    .filter(col("invoice_id") == "INV000001")

print("All line items for invoice INV000001:")
df_invoice_lines.show(truncate=False)

print(f"\nTotal line items: {df_invoice_lines.count()}")

# Calculate total without considering discount
print("\nCalculating totals:")
for row in df_invoice_lines.collect():
    line_total_before = row.quantity * row.unit_price
    if row.discount is not None:
        line_total_after = line_total_before * (1 - row.discount)
        print(f"{row.invoice_line_id}: {row.quantity} × {row.unit_price} × (1 - {row.discount}) = {line_total_after:.2f}")
    else:
        print(f"{row.invoice_line_id}: {row.quantity} × {row.unit_price} × (1 - NULL) = {line_total_before:.2f} (discount unknown)")

# Get payment amount for this invoice
df_payment = spark.table("bronze_catalog.bronze_stage_schema.payments") \
    .filter(col("invoice_id") == "INV000001")

print("\nPayment for INV000001:")
df_payment.show(truncate=False)

In [0]:
from pyspark.sql.functions import col

print("=" * 80)
print("COMPLETE ANALYSIS: NULL DISCOUNT CALCULATION FOR INV000001")
print("=" * 80)

# Step 1: Get invoice currency
df_invoice = spark.table("bronze_catalog.bronze_stage_schema.invoices") \
    .filter(col("invoice_id") == "INV000001")
invoice_currency = df_invoice.select("currency").first()[0]
print(f"\n1. INVOICE CURRENCY: {invoice_currency}")
print("   → All line items and payment are in this currency")

# Step 2: Get line items
df_lines = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items") \
    .filter(col("invoice_id") == "INV000001").collect()

print(f"\n2. LINE ITEMS (Currency: {invoice_currency}):")
for line in df_lines:
    print(f"   {line.invoice_line_id}: Product {line.product_id}, Qty {line.quantity}, Price {line.unit_price}, Discount {line.discount}")

# Step 3: Get payment amount
df_payment = spark.table("bronze_catalog.bronze_stage_schema.payments") \
    .filter(col("invoice_id") == "INV000001")
payment_amount = df_payment.select("payment_amount").first()[0]
print(f"\n3. PAYMENT AMOUNT: {payment_amount} {invoice_currency}")

# Step 4: Calculate the NULL discount
print("\n4. CALCULATION:")
print("   " + "-" * 70)

# L0000002 (has discount)
qty_2 = 8
price_2 = 337
discount_2 = 0.05
total_2 = qty_2 * price_2 * (1 - discount_2)
print(f"   L0000002: {qty_2} × {price_2} × (1 - {discount_2}) = {total_2:.2f} {invoice_currency}")

# L0000001 (NULL discount - to calculate)
qty_1 = 6
price_1 = 375
total_1_needed = payment_amount - total_2
print(f"   L0000001: Must equal {payment_amount} - {total_2:.2f} = {total_1_needed:.2f} {invoice_currency}")

# Calculate discount
total_before_discount = qty_1 * price_1
print(f"   L0000001 before discount: {qty_1} × {price_1} = {total_before_discount} {invoice_currency}")

discount_multiplier = total_1_needed / total_before_discount
discount_1 = 1 - discount_multiplier

print("\n5. RESULT:")
print("   " + "=" * 70)
print(f"   NULL discount for L0000001 should be: {discount_1:.4f} or {discount_1*100:.2f}%")
print("   " + "=" * 70)

# Step 6: Verification
print("\n6. VERIFICATION:")
verify_1 = qty_1 * price_1 * (1 - discount_1)
verify_2 = total_2
verify_total = verify_1 + verify_2

print(f"   L0000001: {qty_1} × {price_1} × (1 - {discount_1:.4f}) = {verify_1:.2f} {invoice_currency}")
print(f"   L0000002: {verify_2:.2f} {invoice_currency}")
print(f"   Total: {verify_1:.2f} + {verify_2:.2f} = {verify_total:.2f} {invoice_currency}")
print(f"   Expected payment: {payment_amount} {invoice_currency}")
print(f"   Difference: {abs(verify_total - payment_amount):.2f} {invoice_currency}")

if abs(verify_total - payment_amount) < 1:
    print("\n   ✓ CORRECT! The calculation matches the payment amount.")
else:
    print("\n   ⚠ Mismatch detected.")

print("\n" + "=" * 80)
print("CONCLUSION: Replace NULL discount in L0000001 with 0.55 (or 55%)")
print("=" * 80)

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.invoices")
 
# Display the data
display(df_customers)

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.invoices where invoice_id ='INV000004';

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.payments")
 
# Display the data
display(df_customers)

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.payments where invoice_id ='INV000004';

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.products")
 
# Display the data
display(df_customers)

In [0]:
%sql
select * from bronze_catalog.bronze_stage_schema.products where product_id in("P007","P010","P023","P032","P023","P058");

In [0]:
df_customers = spark.table("bronze_catalog.bronze_stage_schema.regions")
 
# Display the data
display(df_customers)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when, coalesce, first, max as spark_max, lit
from pyspark.sql.window import Window

print("=" * 80)
print("CALCULATING NULL DISCOUNTS FOR ALL INVOICE LINE ITEMS WITH CURRENCY CONVERSION")
print("=" * 80)

# Read all required tables
df_line_items = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")
df_invoices = spark.table("bronze_catalog.bronze_stage_schema.invoices")
df_payments = spark.table("bronze_catalog.bronze_stage_schema.payments")
df_exchange_rates = spark.table("bronze_catalog.bronze_stage_schema.exchange_rates")

# Get the latest exchange rate for each currency (using max date)
df_latest_rates = df_exchange_rates.groupBy("currency").agg(
    spark_max("date").alias("max_date")
)
df_rates = df_exchange_rates.alias("er").join(
    df_latest_rates.alias("lr"),
    (col("er.currency") == col("lr.currency")) & (col("er.date") == col("lr.max_date"))
).select(
    col("er.currency"),
    col("er.rate_to_usd").alias("rate")
).distinct()

print("\n1. EXCHANGE RATES (to USD):")
df_rates.show()

# Step 1: Join line items with invoices to get currency
df_line_with_invoice = df_line_items.alias("li").join(
    df_invoices.alias("inv"),
    col("li.invoice_id") == col("inv.invoice_id")
)

# Step 2: Join with exchange rates to convert to USD
df_line_with_rates = df_line_with_invoice.join(
    df_rates.alias("rates"),
    col("inv.currency") == col("rates.currency"),
    "left"
)

# Step 3: Calculate line totals in USD
df_line_with_totals = df_line_with_rates.withColumn(
    "line_total_local",
    col("li.quantity") * col("li.unit_price") * (1 - coalesce(col("li.discount"), lit(0)))
).withColumn(
    "line_total_usd",
    col("line_total_local") * col("rates.rate")
).withColumn(
    "has_null_discount",
    when(col("li.discount").isNull(), 1).otherwise(0)
)

# Step 4: Join payments and convert to USD
df_payments_with_invoice = df_payments.alias("pay").join(
    df_invoices.alias("inv2").select(
        col("inv2.invoice_id"),
        col("inv2.currency").alias("invoice_currency")
    ),
    col("pay.invoice_id") == col("inv2.invoice_id")
)

df_payments_with_rates = df_payments_with_invoice.join(
    df_rates.alias("rates2"),
    col("invoice_currency") == col("rates2.currency"),
    "left"
).withColumn(
    "payment_amount_usd",
    col("pay.payment_amount") * col("rates2.rate")
)

# Step 5: Calculate totals by invoice
df_invoice_summary = df_line_with_totals.groupBy(col("li.invoice_id").alias("invoice_id")).agg(
    spark_sum("line_total_usd").alias("total_lines_usd"),
    spark_sum("has_null_discount").alias("null_discount_count"),
    first(col("inv.currency")).alias("invoice_currency")
)

# Join with payment amounts
df_invoice_with_payment = df_invoice_summary.alias("summary").join(
    df_payments_with_rates.alias("paym").select(
        col("paym.invoice_id").alias("payment_invoice_id"),
        col("paym.payment_amount").alias("payment_amount"),
        col("paym.payment_amount_usd")
    ),
    col("summary.invoice_id") == col("payment_invoice_id"),
    "left"
)

# Step 6: Calculate discrepancy
df_invoice_analysis = df_invoice_with_payment.withColumn(
    "discrepancy_usd",
    col("payment_amount_usd") - col("total_lines_usd")
).filter(col("null_discount_count") > 0)

print("\n2. INVOICES WITH NULL DISCOUNTS:")
print(f"Total invoices with NULL discounts: {df_invoice_analysis.count()}")

print("\n3. SAMPLE ANALYSIS (First 10 invoices):")
df_invoice_analysis.select(
    "invoice_id",
    "invoice_currency",
    "null_discount_count",
    "payment_amount",
    "payment_amount_usd",
    "total_lines_usd",
    "discrepancy_usd"
).show(10, truncate=False)

# Step 7: For invoices with EXACTLY 1 NULL discount, we can calculate the discount
df_single_null = df_invoice_analysis.filter(col("null_discount_count") == 1)

print(f"\n4. INVOICES WITH EXACTLY 1 NULL DISCOUNT (calculable): {df_single_null.count()}")

if df_single_null.count() > 0:
    print("\nSample calculations for invoices with 1 NULL discount:")
    
    # Get the line items with NULL discount for these invoices
    invoice_ids = [row.invoice_id for row in df_single_null.select("invoice_id").limit(5).collect()]
    
    for inv_id in invoice_ids:
        print(f"\n{'='*70}")
        print(f"Invoice: {inv_id}")
        print(f"{'='*70}")
        
        # Get line items for this invoice
        lines = df_line_items.filter(col("invoice_id") == inv_id).collect()
        invoice = df_invoices.filter(col("invoice_id") == inv_id).first()
        payment = df_payments.filter(col("invoice_id") == inv_id).first()
        
        if payment:
            print(f"Currency: {invoice.currency}")
            print(f"Payment: {payment.payment_amount} {invoice.currency}")
            print(f"\nLine Items:")
            
            total_known = 0
            null_line = None
            
            for line in lines:
                if line.discount is not None:
                    line_total = line.quantity * line.unit_price * (1 - line.discount)
                    total_known += line_total
                    print(f"  {line.invoice_line_id}: {line.quantity} × {line.unit_price} × (1-{line.discount}) = {line_total:.2f}")
                else:
                    null_line = line
                    print(f"  {line.invoice_line_id}: {line.quantity} × {line.unit_price} × (1-NULL) = ?")
            
            if null_line:
                # Calculate required total for NULL line
                required_total = payment.payment_amount - total_known
                before_discount = null_line.quantity * null_line.unit_price
                
                if before_discount > 0:
                    calculated_discount = 1 - (required_total / before_discount)
                    
                    print(f"\nCalculation:")
                    print(f"  Required total for {null_line.invoice_line_id}: {payment.payment_amount} - {total_known:.2f} = {required_total:.2f}")
                    print(f"  Before discount: {null_line.quantity} × {null_line.unit_price} = {before_discount}")
                    print(f"  → Calculated discount: {calculated_discount:.4f} ({calculated_discount*100:.2f}%)")
                    
                    # Verify
                    verify_total = (before_discount * (1 - calculated_discount)) + total_known
                    print(f"\nVerification: {verify_total:.2f} ≈ {payment.payment_amount} {invoice.currency}")
                    if abs(verify_total - payment.payment_amount) < 1:
                        print("  ✓ Match!")
                    else:
                        print(f"  ⚠ Mismatch: {abs(verify_total - payment.payment_amount):.2f}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Total invoices analyzed: {df_invoice_analysis.count()}")
print(f"Invoices with exactly 1 NULL discount (calculable): {df_single_null.count()}")
print(f"Invoices with multiple NULL discounts (need business rules): {df_invoice_analysis.filter(col('null_discount_count') > 1).count()}")
print("\nNOTE: For invoices with multiple NULL discounts, business rules are needed.")
print("Possible approaches:")
print("  1. Replace all NULL with 0 (no discount)")
print("  2. Distribute discrepancy proportionally")
print("  3. Set a default discount rate")
print("=" * 80)